In [1]:
import os
import re
import requests
import subprocess
from pathlib import Path
from tqdm import tqdm


def get_direct_file_link(mailru_file_url: str) -> str:
    """
    Преобразует публичную ссылку вида:
        https://cloud.mail.ru/public/<key>/<subkey>/<filename>
    в прямую ссылку на CDN, по которой можно скачать файл через wget или requests.

    Возвращает прямую ссылку для скачивания.
    """
    resp = requests.get(mailru_file_url)
    if resp.status_code != 200:
        raise RuntimeError(f"Ошибка {resp.status_code} при запросе {mailru_file_url}")

    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', resp.text)
    if not match:
        raise RuntimeError("Не удалось найти CDN ссылку в HTML Mail.ru")

    base_url = match.group(1)
    parts = mailru_file_url.strip("/").split("/")[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"


def download_from_mailru(file_url: str, local_name: str, force: bool = False, show_progress: bool = True):
    """
    Скачивает файл с Mail.ru по публичной ссылке.

    Args:
        file_url: ссылка на файл в облаке Mail.ru.
        local_name: имя файла для сохранения.
        force: если True — перекачивает даже если файл уже есть.
        show_progress: показывать ли прогресс-бар.
    """
    local_path = Path(local_name)
    if local_path.exists() and not force:
        print(f"Файл {local_name} уже существует, пропускаем скачивание.")
        return

    direct = get_direct_file_link(file_url)
    print(f"Скачиваем {file_url} → {local_name}")

    with requests.get(direct, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        block_size = 8192
        with open(local_name, "wb") as f, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=f"Downloading {local_name}",
            disable=not show_progress,
        ) as bar:
            for chunk in r.iter_content(block_size):
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Файл {local_name} успешно скачан ({os.path.getsize(local_name)/1e6:.1f} MB).")

In [2]:
# Ссылки на данные по задаче
train_link = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/train_data.tar"
test_link  = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar"

In [3]:
# Если скорость загрузки низкая — это может быть связано с CDN.
# Попробуйте перезапустить ячейку: при новом соединении может попасться другой узел CDN,
# и загрузка обычно проходит быстрее (2-3 минуты при нормальном узле).
# download_from_mailru(train_link, "train_data.tar")
download_from_mailru(test_link, "test_data.tar")

Скачиваем https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar → test_data.tar


Файл test_data.tar успешно скачан (741.0 MB).


In [4]:
# Распаковка
# !tar -xvf train_data.tar &> logs.txt
!tar -xvf test_data.tar &> logs.txt

In [5]:
# Удаление macOS-мусора из папки (Kaggle / Linux / macOS / Windows)

from pathlib import Path
import shutil

# УКАЖИТЕ ПАПКУ ДЛЯ ЧИСТКИ:
TARGET_DIR = Path("/kaggle/working/test_opus")  # замените на вашу директорию

CRUFT_DIRS = {
    "__MACOSX",
    ".Trashes",
    ".Spotlight-V100",
    ".fseventsd",
    ".AppleDouble",
    ".AppleDB",
    ".AppleDesktop",
    ".TemporaryItems",
}

CRUFT_FILES = {
    ".DS_Store",
    ".VolumeIcon.icns",
    ".LSOverride",
    ".localized",
    # "Icon\r" — спец. имя, иногда встречается в мак-архивах
}

def is_cruft_file(p: Path) -> bool:
    name = p.name
    if name in CRUFT_FILES:
        return True
    if name.startswith("._"):  # AppleDouble ресурсилки
        return True
    # Поймаем редкий случай "Icon\r"
    try:
        if name == "Icon\r":
            return True
    except Exception:
        pass
    return False

def clean_macos_junk(root: Path) -> int:
    if not root.exists():
        print(f"Папка не найдена: {root}")
        return 0

    deleted = 0

    # Если сама целевая папка — мусорная (редко, но возможно)
    if root.is_dir() and root.name in CRUFT_DIRS:
        try:
            shutil.rmtree(root, ignore_errors=True)
            print(f"Удалена папка: {root}")
            return 1
        except Exception as e:
            print(f"[WARN] Не удалось удалить {root}: {e}")

    # Обходим глубиной от более глубоких путей к верхним
    for p in sorted(root.rglob("*"), key=lambda x: len(x.parts), reverse=True):
        try:
            if p.is_dir() and p.name in CRUFT_DIRS:
                shutil.rmtree(p, ignore_errors=True)
                deleted += 1
            elif p.is_file() and is_cruft_file(p):
                # Python 3.8+: missing_ok не везде, поэтому try/except
                try:
                    p.unlink()
                except FileNotFoundError:
                    pass
                deleted += 1
        except Exception as e:
            print(f"[WARN] Пропуск {p}: {e}")

    return deleted

removed = clean_macos_junk(TARGET_DIR)
print(f"Готово. Удалено объектов: {removed}")

Готово. Удалено объектов: 27001


In [6]:
!git clone https://github.com/salute-developers/GigaAM.git
%cd GigaAM
!pip install -e .

Cloning into 'GigaAM'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 74 (delta 21), reused 12 (delta 12), pack-reused 41 (from 1)
Receiving objects: 100% (74/74), 1.56 MiB | 2.57 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/kaggle/working/GigaAM
Obtaining file:///kaggle/working/GigaAM
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 918.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.5 MB/s eta 0:00:00
   ━━

In [7]:
import os
import re
import csv
import tarfile
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Optional, List

import torch
import gigaam

ROOT_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_CSV = WORK_DIR / "submition.csv"

# Регулярки для целевых фраз (инвариант к регистру, допускаем пунктуацию между словами)
TARGET_REGEX = re.compile(r"\bне\W*слышу\b|\bне\W*слышно\b", flags=re.IGNORECASE)

def log(*args):
    print(*args, flush=True)

def ensure_ffmpeg() -> bool:
    return shutil.which("ffmpeg") is not None

def extract_if_needed() -> Optional[Path]:
    """
    Ищет test_data.tar под /kaggle/input и распаковывает в /kaggle/working/test_data,
    либо возвращает путь к существующей папке test_data.
    """
    # 1) Если уже распаковано ранее
    ready = WORK_DIR / "test_opus"
    if ready.is_dir():
        return ready

    # 2) Ищем tar
    tars: List[Path] = []
    for d in ROOT_INPUT.iterdir():
        if d.is_dir():
            cand = d / "test_data.tar"
            if cand.exists():
                tars.append(cand)
    if tars:
        td = WORK_DIR / "test_data"
        td.mkdir(exist_ok=True, parents=True)
        log(f"Found test_data.tar at: {tars[0]}. Extracting to {td} ...")
        with tarfile.open(tars[0], "r") as tf:
            tf.extractall(td)
        return td

    # 3) Ищем уже готовую папку test_data
    for d in ROOT_INPUT.iterdir():
        if d.is_dir() and (d / "test_data").is_dir():
            return d / "test_data"
        if d.is_dir() and (d.name.lower().startswith("test") or d.name.lower().endswith("test")):
            # На случай, если папка названа иначе, но внутри есть audio/
            if (d / "audio").is_dir():
                return d

    # 4) В качестве fallback ищем любую папку с audio/
    for d in ROOT_INPUT.iterdir():
        if d.is_dir() and (d / "audio").is_dir():
            return d

    return None

def find_audio_dir(root_test: Path) -> Optional[Path]:
    """
    Возвращает путь к папке с .opus файлами.
    Ожидаем test_data/audio, но делаем поиск на всякий случай.
    """
    direct = root_test / "audio"
    if direct.is_dir():
        return direct
    # Поиск глубже
    for p in root_test.rglob("*"):
        if p.is_dir() and p.name == "audio":
            return p
    return None

def decode_to_wav_16k_mono(input_path: Path) -> Path:
    """
    Декодируем .opus в 16 кГц mono WAV через ffmpeg во временную папку.
    Возвращаем путь к WAV.
    """
    tmp_dir = WORK_DIR / "tmp_wav"
    tmp_dir.mkdir(exist_ok=True, parents=True)
    out_wav = tmp_dir / f"{input_path.stem}_16k.wav"
    cmd = [
        "ffmpeg", "-nostdin", "-y",
        "-i", str(input_path),
        "-ac", "1",
        "-ar", "16000",
        "-vn",
        "-f", "wav",
        str(out_wav),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_wav

def transcribe_file(model, path_opus: Path) -> str:
    """
    Транскрибирует один файл: .opus -> 16k WAV -> ASR.
    """
    wav = decode_to_wav_16k_mono(path_opus)
    try:
        text = model.transcribe(str(wav)) or ""
    finally:
        # Оставляем кэш WAV до конца сессии (ускоряет повторные запуски).
        pass
    return text

def contains_phrase(text: str) -> int:
    norm = " ".join(text.lower().split())
    return 1 if TARGET_REGEX.search(norm) else 0

def main():
    log("Checking ffmpeg ...")
    if not ensure_ffmpeg():
        raise RuntimeError("ffmpeg не найден в окружении Kaggle.")

    log("Locating / extracting test data ...")
    test_root = extract_if_needed()
    if test_root is None:
        raise FileNotFoundError("Не удалось найти test_data(.tar) или папку с audio/ под /kaggle/input.")

    audio_dir = find_audio_dir(test_root)
    if audio_dir is None:
        raise FileNotFoundError(f"В {test_root} не найдена папка audio с тестовыми .opus файлами.")
    log(f"Test audio dir: {audio_dir}")

    files = sorted(audio_dir.glob("*.opus"))
    if not files:
        raise FileNotFoundError(f"В {audio_dir} нет .opus файлов.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "v2_rnnt"  # GigaAM-RNNT v2
    log(f"Loading GigaAM model: {model_name} on {device} ...")
    model = gigaam.load_model(model_name)
    model.to(device)

    rows = [("id", "label", 'text')]
    for i, f in tqdm(enumerate(files, 1), total=len(files)):
        rec_id = f.stem
        try:
            text = transcribe_file(model, f)
            label = contains_phrase(text)
        except Exception as e:
            log(f"[WARN] {rec_id}: {e}")
            label = 0
        rows.append((rec_id, str(label), text))

        # if i % 25 == 0 or i == len(files):
        #     log(f"Processed {i}/{len(files)}")

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as w:
        csv.writer(w).writerows(rows)

    log(f"Saved submission to: {OUT_CSV}")
    # Показать первые строки
    with open(OUT_CSV, "r", encoding="utf-8") as r:
        for k, line in zip(range(5), r):
            print(line.rstrip())

if __name__ == "__main__":
    main()

Checking ffmpeg ...
Locating / extracting test data ...
Test audio dir: /kaggle/working/test_opus/audio
Loading GigaAM model: v2_rnnt on cuda ...


100%|███████████████████████████████████████| 892M/892M [00:42<00:00, 22.2MiB/s]
/kaggle/working/GigaAM/gigaam/__init__.py:118: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

Saved submission to: /kaggle/working/submition.csv
id,label,text
0000219778122723066859323624505982384475,0,дату указывай внизу мы с вами встретимся
0000920560142346477464477964040846645823,1,меня пугает что я не слышу звука в игре
0002106775361063830068199242310438122126,1,срем воду включили чтобы не слышно было и вот так вот тревожимся мы
0002161736146841817059430282255903999813,0,это нарушение липидного профиля как одна из причин


In [8]:
!rm -rf test_opus
!rm -rf GigaAM
!rm -rf tmp_wav